# Time Series Course — m11-fl1 (block 3)

_From the **Time Series & Forecasting** course on Zylo._

Run all cells (`Runtime → Run all` or `Ctrl+F9`) to execute.

In [ ]:
import torch
import torch.nn as nn
import math

class TransformerForecaster(nn.Module):
    def __init__(self, d_model=32, nhead=4, num_layers=2,
                 seq_len=168, forecast_horizon=24):
        super().__init__()
        self.input_proj = nn.Linear(1, d_model)
        pe = torch.zeros(seq_len, d_model)
        pos = torch.arange(seq_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float()
                        * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))

        layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead,
            dim_feedforward=d_model*4, dropout=0.1, batch_first=True)
        self.encoder = nn.TransformerEncoder(layer, num_layers=num_layers)
        self.head = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.ReLU(),
            nn.Linear(d_model, forecast_horizon))

    def forward(self, x):
        x = self.input_proj(x) + self.pe[:, :x.size(1), :]
        x = self.encoder(x)
        return self.head(x.mean(dim=1))

# Training loop
model = TransformerForecaster()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.MSELoss()

# Normalize
mu, sigma = X_train.mean(), X_train.std()
X_tr = torch.FloatTensor((X_train - mu) / sigma).unsqueeze(-1)
y_tr = torch.FloatTensor((y_train - mu) / sigma)
X_te = torch.FloatTensor((X_test - mu) / sigma).unsqueeze(-1)
y_te = torch.FloatTensor((y_test - mu) / sigma)

for epoch in range(30):
    model.train()
    pred = model(X_tr)
    loss = criterion(pred, y_tr)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    if (epoch+1) % 10 == 0:
        model.eval()
        with torch.no_grad():
            test_loss = criterion(model(X_te), y_te)
        print(f"Epoch {epoch+1}: train={loss:.4f}, test={test_loss:.4f}")